# Day 5: 数据质量决定一切

> "Less is more" — LIMA 论文的核心发现

学习目标：
1. 理解 LIMA 论文的核心发现
2. 掌握高质量 SFT 数据的 5 个标准
3. 识别低质量数据的常见问题
4. 学会用 LLM API 生成训练数据
5. 通过实验验证：少量高质量 > 大量低质量

## 1. LIMA 论文核心发现

**LIMA: Less Is More for Alignment**（2023，Meta）

### 核心结论
- 仅用 **1000 条**精心挑选的高质量数据做 SFT
- 效果就能接近甚至超过用数万条数据训练的模型
- 说明：模型的知识主要来自预训练，SFT 只是教它"格式"

### 关键启示
1. 数据质量 >> 数据数量
2. 预训练已经学到了大部分知识
3. SFT 的作用是"对齐"而非"教知识"
4. 精心策展的小数据集比大规模低质量数据更有效

## 2. 高质量数据的 5 个标准

| 标准 | 说明 | 反面 |
|------|------|------|
| **准确性** | 信息正确无误 | 事实错误、过时信息 |
| **多样性** | 覆盖不同主题和难度 | 同质化、重复 |
| **复杂度** | 有适当的深度和细节 | 过于简单、敷衍 |
| **格式规范** | 结构清晰、排版一致 | 杂乱无章 |
| **覆盖度** | 覆盖目标应用场景 | 偏科、盲区 |

## 3. 好数据 vs 坏数据对比

In [ ]:
import json

# 高质量数据示例
high_quality_example = {
    "instruction": "解释什么是梯度下降算法？",
    "output": """梯度下降是一种迭代优化算法，用于找到函数的最小值。其核心思想是：

1. **基本原理**：沿着函数梯度的反方向移动，因为梯度指向函数增长最快的方向。

2. **更新公式**：θ = θ - α × ∇J(θ)
   - α 是学习率
   - ∇J(θ) 是损失函数的梯度

3. **常见变体**：
   - 批量梯度下降（BGD）：使用全部数据，稳定但慢
   - 随机梯度下降（SGD）：每次用一个样本，快但不稳定
   - Mini-batch SGD：折中方案，最常用

4. **学习率的重要性**：太大会发散，太小收敛慢。Adam 等方法可自适应调整。"""
}

# 低质量数据示例
low_quality_example = {
    "instruction": "解释什么是梯度下降算法？",
    "output": "梯度下降就是一种算法。"
}

print("高质量数据:")
print(json.dumps(high_quality_example, ensure_ascii=False, indent=2))
print(f"\n输出长度: {len(high_quality_example['output'])} 字符")
print("\n" + "=" * 60)
print("\n低质量数据:")
print(json.dumps(low_quality_example, ensure_ascii=False, indent=2))
print(f"\n输出长度: {len(low_quality_example['output'])} 字符")

## 4. 常见的坏数据问题

In [ ]:
# 坏数据的典型表现
bad_data_examples = [
    {
        "问题类型": "过于简短",
        "instruction": "什么是 Docker？",
        "output": "Docker 是一个工具。",
        "问题": "缺乏任何有用的信息"
    },
    {
        "问题类型": "事实错误",
        "instruction": "Python 是编译型还是解释型语言？",
        "output": "Python 是编译型语言，运行前需要编译成机器码。",
        "问题": "Python 是解释型语言"
    },
    {
        "问题类型": "格式混乱",
        "instruction": "列出 3 种排序算法",
        "output": "冒泡排序快速排序归并排序它们都是排序算法各有优缺点",
        "问题": "没有分隔、没有结构"
    },
    {
        "问题类型": "答非所问",
        "instruction": "如何部署一个 Flask 应用？",
        "output": "Flask 是一个 Python Web 框架，由 Armin Ronacher 创建。",
        "问题": "没有回答部署问题"
    },
]

for i, example in enumerate(bad_data_examples, 1):
    print(f"\n坏数据 #{i}: {example['问题类型']}")
    print(f"  Q: {example['instruction']}")
    print(f"  A: {example['output']}")
    print(f"  问题: {example['问题']}")
    print("-" * 50)

## 5. 用 LLM API 生成高质量数据

以下是一个数据生成的代码框架（需要 API key 才能运行）。

In [ ]:
# 数据生成模板（框架代码，展示思路）

GENERATION_PROMPT = """
你是一位资深技术专家。请针对以下问题给出高质量的回答。

要求：
1. 回答准确、有深度
2. 使用清晰的结构（标题、列表、代码示例等）
3. 包含实际案例或最佳实践
4. 适合初学者理解
5. 300-500 字

问题：{instruction}
"""

# 质量检查清单
QUALITY_CHECKLIST = [
    "事实准确性：答案中的信息是否正确？",
    "完整性：是否覆盖了问题的主要方面？",
    "结构性：是否有清晰的组织结构？",
    "可读性：是否易于理解？",
    "一致性：风格和格式是否统一？",
]

print("数据生成 Prompt 模板:")
print(GENERATION_PROMPT)
print("\n质量检查清单:")
for item in QUALITY_CHECKLIST:
    print(f"  [ ] {item}")

print("\n--- 使用示例（伪代码）---")
print("""
import openai
client = openai.OpenAI(api_key="your-key")

topics = ["什么是微服务？", "解释 REST API", ...]
for topic in topics:
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": GENERATION_PROMPT.format(instruction=topic)}],
        temperature=0.7,
    )
    answer = response.choices[0].message.content
    # 人工审核后加入训练集
""")

## 6. 对比实验：高质量 vs 低质量

### 6.1 构造实验数据

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2-0.5B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 高质量数据（10条，内容丰富详细）
HIGH_QUALITY_DATA = [
    {"instruction": "解释什么是梯度下降算法？",
     "output": "梯度下降是一种迭代优化算法，用于找到函数的最小值。\n\n1. **基本原理**：沿着梯度反方向移动\n2. **更新公式**：θ = θ - α × ∇J(θ)\n3. **变体**：BGD（全量）、SGD（随机）、Mini-batch（最常用）\n4. **学习率**：太大会震荡，太小收敛慢。Adam 等方法可自适应调整。"},
    {"instruction": "Python 中列表和元组有什么区别？",
     "output": "列表和元组的主要区别：\n\n1. **可变性**：列表可变（mutable），元组不可变（immutable）\n2. **语法**：列表用 `[]`，元组用 `()`\n3. **性能**：元组更快、占内存更少\n4. **使用场景**：列表用于需要修改的集合，元组用于固定数据如坐标 (x,y)\n5. **字典键**：元组可以做字典键，列表不行"},
    {"instruction": "什么是过拟合？如何防止？",
     "output": "过拟合是模型在训练集上表现好但泛化差的现象。\n\n**判断方法**：训练loss下降但验证loss上升\n\n**防止方法**：\n1. 增加数据量\n2. 数据增强\n3. 正则化（L1/L2、Dropout）\n4. 早停（Early Stopping）\n5. 降低模型复杂度\n6. Batch Normalization\n7. 交叉验证"},
    {"instruction": "什么是 Docker？",
     "output": "Docker 是容器化技术，将应用和依赖打包成标准化单元。\n\n**核心概念**：\n1. 镜像（Image）：只读模板\n2. 容器（Container）：镜像的运行实例\n3. Dockerfile：构建脚本\n\n**用途**：环境一致性、快速部署、微服务架构、CI/CD\n\n**vs 虚拟机**：Docker共享OS内核，更轻量，秒级启动"},
    {"instruction": "什么是 RESTful API？",
     "output": "RESTful API 是基于 REST 架构风格的 Web API。\n\n**核心原则**：\n1. 资源导向：URL表示资源，如 `/users/123`\n2. HTTP方法语义化：GET获取、POST创建、PUT更新、DELETE删除\n3. 无状态：每个请求包含所有必要信息\n\n**最佳实践**：\n- URL用名词复数：`/api/users`\n- 用HTTP状态码表示结果\n- 支持分页和版本控制"},
    {"instruction": "什么是递归？",
     "output": "递归是函数调用自身来解决问题的编程技巧。\n\n**核心要素**：\n1. 基线条件（Base Case）：终止递归\n2. 递归步骤：分解为更小的子问题\n\n```python\ndef factorial(n):\n    if n <= 1: return 1\n    return n * factorial(n-1)\n```\n\n**优缺点**：代码简洁但有栈溢出风险\n**应用**：树遍历、快速排序、分治算法"},
    {"instruction": "什么是 Git？",
     "output": "Git 是分布式版本控制系统。\n\n**核心概念**：\n1. 仓库（Repository）\n2. 提交（Commit）：代码快照\n3. 分支（Branch）：独立开发线\n4. 合并（Merge）\n\n**常用命令**：clone, add, commit, push, pull, branch, merge\n\n**分布式优势**：每个开发者都有完整仓库副本"},
    {"instruction": "什么是多线程？",
     "output": "多线程是并发编程模型，允许程序同时执行多个线程。\n\n**概念**：线程是进程内的执行单元，共享内存空间\n\n**挑战**：\n1. 竞态条件：多线程修改共享数据\n2. 死锁：互相等待\n3. Python GIL：限制CPU密集型并行\n\n**选择**：I/O密集用threading/asyncio，CPU密集用multiprocessing"},
    {"instruction": "HTTP 和 HTTPS 的区别？",
     "output": "核心区别在于安全性：\n\n1. **加密**：HTTP明文传输，HTTPS用TLS/SSL加密\n2. **端口**：HTTP用80，HTTPS用443\n3. **证书**：HTTPS需要SSL证书验证身份\n4. **性能**：HTTPS有少量加解密开销\n5. **SEO**：搜索引擎优先收录HTTPS\n\n建议：所有网站都应使用HTTPS"},
    {"instruction": "什么是数据库索引？",
     "output": "索引是加速数据库查询的数据结构，类似书籍目录。\n\n**类型**：B-Tree（最常用）、Hash、全文索引、复合索引\n\n**使用建议**：\n- 在WHERE/JOIN/ORDER BY常用列建索引\n- 不要过度索引\n- 低区分度列不适合单独建索引\n\n**代价**：增加写操作开销，占用额外磁盘空间"},
]

# 低质量数据（15条，回答敷衍）
LOW_QUALITY_DATA = [
    {"instruction": "解释什么是梯度下降算法？", "output": "梯度下降就是一种算法。"},
    {"instruction": "Python 中列表和元组有什么区别？", "output": "列表和元组差不多，一个用方括号一个用圆括号。"},
    {"instruction": "什么是过拟合？如何防止？", "output": "过拟合就是训练过头了，用正则化可以解决。"},
    {"instruction": "什么是 Docker？", "output": "Docker 是一个工具，用来运行程序的。"},
    {"instruction": "什么是 RESTful API？", "output": "就是一种 API 风格。"},
    {"instruction": "什么是递归？", "output": "递归就是函数调用自己。"},
    {"instruction": "什么是 Git？", "output": "Git 是版本管理工具。"},
    {"instruction": "什么是多线程？", "output": "多线程就是同时运行多个线程。"},
    {"instruction": "HTTP 和 HTTPS 的区别？", "output": "HTTPS 比 HTTP 多了个 S，更安全。"},
    {"instruction": "什么是数据库索引？", "output": "索引能让查询变快。"},
    {"instruction": "什么是面向对象编程？", "output": "面向对象就是用类和对象来编程。"},
    {"instruction": "Python 的装饰器是什么？", "output": "装饰器就是在函数上面加 @ 符号。"},
    {"instruction": "什么是 SQL 注入？", "output": "SQL 注入是一种攻击。"},
    {"instruction": "TCP 和 UDP 的区别？", "output": "TCP 可靠，UDP 不可靠。"},
    {"instruction": "什么是缓存？", "output": "缓存就是把数据放在内存里。"},
]

print(f"高质量数据: {len(HIGH_QUALITY_DATA)} 条")
print(f"低质量数据: {len(LOW_QUALITY_DATA)} 条")
print(f"\n高质量平均输出长度: {sum(len(d['output']) for d in HIGH_QUALITY_DATA)/len(HIGH_QUALITY_DATA):.0f} 字符")
print(f"低质量平均输出长度: {sum(len(d['output']) for d in LOW_QUALITY_DATA)/len(LOW_QUALITY_DATA):.0f} 字符")

### 6.2 准备训练数据

In [ ]:
def format_to_chat(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "user", "content": inst},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

hq_dataset = Dataset.from_list(HIGH_QUALITY_DATA)
lq_dataset = Dataset.from_list(LOW_QUALITY_DATA)

hq_dataset = hq_dataset.map(format_to_chat, batched=True, remove_columns=hq_dataset.column_names)
lq_dataset = lq_dataset.map(format_to_chat, batched=True, remove_columns=lq_dataset.column_names)

print(f"高质量数据集: {len(hq_dataset)} 条")
print(f"低质量数据集: {len(lq_dataset)} 条")

### 6.3 训练函数

In [ ]:
def train_model(train_dataset, experiment_name, num_epochs=5):
    """训练一个模型并返回结果"""
    print(f"\n--- 训练: {experiment_name} ---")
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    
    sft_config = SFTConfig(
        output_dir=f"./output_{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4, lr_scheduler_type="cosine",
        warmup_ratio=0.1, logging_steps=5,
        save_strategy="no", max_seq_length=512,
        fp16=torch.cuda.is_available(), report_to="none",
        dataset_text_field="text",
    )
    
    trainer = SFTTrainer(
        model=model, args=sft_config,
        train_dataset=train_dataset, processing_class=tokenizer,
    )
    
    start_time = time.time()
    result = trainer.train()
    elapsed = time.time() - start_time
    
    print(f"  Loss: {result.metrics['train_loss']:.4f}, 耗时: {elapsed:.0f}s")
    return model, result.metrics['train_loss'], elapsed

### 6.4 执行对比实验

In [ ]:
# 训练两个模型
hq_model, hq_loss, hq_time = train_model(hq_dataset, "high_quality")
lq_model, lq_loss, lq_time = train_model(lq_dataset, "low_quality")

print(f"\n训练结果:")
print(f"  高质量: loss={hq_loss:.4f}, 耗时={hq_time:.0f}s")
print(f"  低质量: loss={lq_loss:.4f}, 耗时={lq_time:.0f}s")

### 6.5 自动评估对比

In [ ]:
def generate_response(model, prompt, max_new_tokens=200):
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def simple_quality_score(response):
    """简单质量评分（0-100）"""
    score = 0
    # 长度评分
    length = len(response)
    if length > 300: score += 30
    elif length > 150: score += 20
    elif length > 50: score += 10
    # 结构评分
    if "\n" in response: score += 10
    if any(m in response for m in ["1.", "2.", "- ", "**"]): score += 10
    if "```" in response: score += 10
    # 内容深度
    keywords = ["例如", "比如", "原因", "优点", "缺点", "原理", "步骤", "建议", "场景"]
    score += min(sum(1 for kw in keywords if kw in response) * 8, 40)
    return min(score, 100)

In [ ]:
# 评估
eval_prompts = [
    "什么是梯度下降？请简要说明。",
    "解释 Python 中 list 和 tuple 的区别。",
    "什么是微服务架构？",
    "什么是设计模式中的单例模式？",
]

print("对比评估结果")
print("=" * 80)
hq_total, lq_total = 0, 0

for prompt in eval_prompts:
    hq_resp = generate_response(hq_model, prompt)
    lq_resp = generate_response(lq_model, prompt)
    hq_score = simple_quality_score(hq_resp)
    lq_score = simple_quality_score(lq_resp)
    hq_total += hq_score
    lq_total += lq_score
    
    print(f"\n问: {prompt}")
    print(f"  [高质量模型] 得分: {hq_score}")
    print(f"  {hq_resp[:200]}")
    print(f"  [低质量模型] 得分: {lq_score}")
    print(f"  {lq_resp[:200]}")
    print("-" * 80)

hq_avg = hq_total / len(eval_prompts)
lq_avg = lq_total / len(eval_prompts)
print(f"\n实验结论:")
print(f"  高质量模型平均得分: {hq_avg:.1f} (数据量: {len(HIGH_QUALITY_DATA)} 条)")
print(f"  低质量模型平均得分: {lq_avg:.1f} (数据量: {len(LOW_QUALITY_DATA)} 条)")
print(f"  得分差异: {hq_avg - lq_avg:+.1f}")

## 7. 数据工程核心原则总结

### 原则一：质量优先
数据质量远比数量重要。10 条精心编写的数据可能比 1000 条低质量数据更有效。

### 原则二：多样性
确保数据覆盖目标场景的各个方面，避免过度集中在某一类问题。

### 原则三：一致性
保持数据格式、风格、长度的一致性，减少模型学习的噪声。

### 原则四：迭代改进
- 先用少量高质量数据训练
- 评估模型弱点
- 针对性地补充数据
- 重复迭代

### 原则五：人机协作
- 用 LLM 生成初稿
- 人工审核和修正
- 构建质量检查 pipeline

### 思考题

1. 如果你要微调一个客服机器人，如何构建高质量训练数据？
2. 数据增强在 NLP 中有哪些有效方法？
3. 如何自动检测训练数据中的低质量样本？
4. LIMA 的结论是否适用于所有场景？什么时候数据量确实很重要？